#### Queens Supermarket Sales ETL Pipeline  
**Client Context:** Midroc Commerce (Queens Supermarket)  

**Objective:**  
Design and implement a dimensional star schema in MySQL suitable for analytics and BI reporting. 
This includes creating dimension tables and the central fact table following data warehousing best practices.

**Key Deliverables:**
- Dimension tables (dim_date, dim_branch, dim_product, dim_customer_type, dim_payment)
- Central Fact table (fact_sales)
- Proper primary keys, foreign keys, and indexes
- Schema documentation

#### Import Libraries

In [8]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import json
from dotenv import load_dotenv
import sqlalchemy as sa
from sqlalchemy import create_engine, text

# Load environment variables
load_dotenv()

print("Libraries imported and .env loaded successfully!")

Libraries imported and .env loaded successfully!


#### Database Connection Setup

In [6]:
load_dotenv()

# Get credentials from .env
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '3306')
DB_NAME = os.getenv('DB_NAME', 'queens_supermarket_dw')

# First, connect without specifying database to create it
base_connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}"

# Create engine without database
engine_base = create_engine(base_connection_string, pool_pre_ping=True)

print("Base connection established (without database)")

Base connection established (without database)


#### Create Database

In [7]:
create_db_query = f"""
CREATE DATABASE IF NOT EXISTS `{DB_NAME}` 
CHARACTER SET utf8mb4 
COLLATE utf8mb4_unicode_ci;
"""

with engine_base.connect() as conn:
    conn.execute(text(create_db_query))
    print(f" Database '{DB_NAME}' created or already exists.")

# Now create engine with the specific database
connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string, pool_pre_ping=True)

# Switch to the database
with engine.connect() as conn:
    conn.execute(text(f"USE `{DB_NAME}`"))

print(f" Successfully connected to database: {DB_NAME}")

 Database 'queens_supermarket_dw' created or already exists.
 Successfully connected to database: queens_supermarket_dw


#### Load Silver Data

In [9]:
SILVER_PATH = '../data/silver/queens_supermarket_sales_silver.parquet'
df_silver = pd.read_parquet(SILVER_PATH)

print(f"Silver data loaded: {df_silver.shape}")

Silver data loaded: (50000, 30)


#### Create Star Schema - Dimension & Fact Tables (DDL)

In [19]:
print(" Dropping existing tables safely...")

with engine.connect() as conn:
    # Disable foreign key checks temporarily
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    
    # Drop tables in correct order (fact table first)
    drop_statements = [
        "DROP TABLE IF EXISTS fact_sales;",
        "DROP TABLE IF EXISTS dim_date;",
        "DROP TABLE IF EXISTS dim_branch;",
        "DROP TABLE IF EXISTS dim_product;",
        "DROP TABLE IF EXISTS dim_customer_type;",
        "DROP TABLE IF EXISTS dim_payment;"
    ]
    
    for stmt in drop_statements:
        conn.execute(text(stmt))
        print(f"   Dropped: {stmt.strip()}")
    
    # Enable foreign key checks back
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))

print("All existing tables dropped successfully.\n")

# ====================== CREATE NEW TABLES ======================
print("Creating Star Schema tables...")

create_statements = [
    """
    CREATE TABLE dim_date (
        date_key INT PRIMARY KEY,
        full_date DATE NOT NULL,
        year INT NOT NULL,
        month INT NOT NULL,
        day INT NOT NULL,
        weekday INT NOT NULL,
        weekday_name VARCHAR(10),
        is_weekend TINYINT(1) DEFAULT 0,
        month_name VARCHAR(15)
    );
    """,
    """
    CREATE TABLE dim_branch (
        branch_key INT AUTO_INCREMENT PRIMARY KEY,
        branch_id VARCHAR(10) NOT NULL UNIQUE,
        branch_name VARCHAR(50) NOT NULL,
        city VARCHAR(50) NOT NULL
    );
    """,
    """
    CREATE TABLE dim_product (
        product_key INT AUTO_INCREMENT PRIMARY KEY,
        product_line VARCHAR(100) NOT NULL
    );
    """,
    """
    CREATE TABLE dim_customer_type (
        customer_type_key INT AUTO_INCREMENT PRIMARY KEY,
        customer_type VARCHAR(20) NOT NULL
    );
    """,
    """
    CREATE TABLE dim_payment (
        payment_key INT AUTO_INCREMENT PRIMARY KEY,
        payment_method VARCHAR(30) NOT NULL
    );
    """,
    """
    CREATE TABLE fact_sales (
        sales_key INT AUTO_INCREMENT PRIMARY KEY,
        invoice_id VARCHAR(20) NOT NULL,
        date_key INT NOT NULL,
        branch_key INT NOT NULL,
        product_key INT NOT NULL,
        customer_type_key INT NOT NULL,
        payment_key INT NOT NULL,
        
        quantity INT NOT NULL,
        unit_price DECIMAL(12,2) NOT NULL,
        subtotal DECIMAL(12,2) NOT NULL,
        tax_15 DECIMAL(12,2) NOT NULL,
        total DECIMAL(12,2) NOT NULL,
        cogs DECIMAL(12,2) NOT NULL,
        gross_income DECIMAL(12,2) NOT NULL,
        gross_margin_percentage DECIMAL(5,2) NOT NULL,
        
        time_of_day VARCHAR(20),
        customer_segment VARCHAR(20),
        high_value_transaction TINYINT(1) DEFAULT 0,
        rating DECIMAL(3,1),
        
        FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
        FOREIGN KEY (branch_key) REFERENCES dim_branch(branch_key),
        FOREIGN KEY (product_key) REFERENCES dim_product(product_key),
        FOREIGN KEY (customer_type_key) REFERENCES dim_customer_type(customer_type_key),
        FOREIGN KEY (payment_key) REFERENCES dim_payment(payment_key),
        
        INDEX idx_date (date_key),
        INDEX idx_branch (branch_key),
        INDEX idx_product (product_key)
    );
    """
]

with engine.connect() as conn:
    for i, stmt in enumerate(create_statements, 1):
        conn.execute(text(stmt))
        print(f"   Created table {i}/6")

print("\n Star Schema created successfully!")

 Dropping existing tables safely...
   Dropped: DROP TABLE IF EXISTS fact_sales;
   Dropped: DROP TABLE IF EXISTS dim_date;
   Dropped: DROP TABLE IF EXISTS dim_branch;
   Dropped: DROP TABLE IF EXISTS dim_product;
   Dropped: DROP TABLE IF EXISTS dim_customer_type;
   Dropped: DROP TABLE IF EXISTS dim_payment;
All existing tables dropped successfully.

Creating Star Schema tables...
   Created table 1/6
   Created table 2/6
   Created table 3/6
   Created table 4/6
   Created table 5/6
   Created table 6/6

 Star Schema created successfully!


#### Populate Dimension Tables

In [27]:
with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    
    # Force drop all dimension tables
    conn.execute(text("DROP TABLE IF EXISTS dim_branch;"))
    conn.execute(text("DROP TABLE IF EXISTS dim_product;"))
    conn.execute(text("DROP TABLE IF EXISTS dim_customer_type;"))
    conn.execute(text("DROP TABLE IF EXISTS dim_payment;"))
    conn.execute(text("DROP TABLE IF EXISTS fact_sales;"))   # Just in case
    
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))

print(" All old dimension tables dropped.")

# Now create tables with proper structure
create_ddl = """
CREATE TABLE dim_branch (
    branch_key INT AUTO_INCREMENT PRIMARY KEY,
    branch_id VARCHAR(10) NOT NULL UNIQUE,
    branch_name VARCHAR(50) NOT NULL,
    city VARCHAR(50) NOT NULL
);

CREATE TABLE dim_product (
    product_key INT AUTO_INCREMENT PRIMARY KEY,
    product_line VARCHAR(100) NOT NULL
);

CREATE TABLE dim_customer_type (
    customer_type_key INT AUTO_INCREMENT PRIMARY KEY,
    customer_type VARCHAR(20) NOT NULL
);

CREATE TABLE dim_payment (
    payment_key INT AUTO_INCREMENT PRIMARY KEY,
    payment_method VARCHAR(30) NOT NULL
);
"""

with engine.connect() as conn:
    for statement in create_ddl.strip().split(';'):
        if statement.strip():
            conn.execute(text(statement.strip() + ";"))

print(" Dimension tables recreated with surrogate keys (AUTO_INCREMENT).")

# ====================== POPULATE THE TABLES ======================

dim_branch = df_silver[['branch_id', 'branch_name', 'city']].drop_duplicates()
dim_branch.to_sql('dim_branch', engine, if_exists='append', index=False)

dim_product = df_silver[['product_line']].drop_duplicates()
dim_product.to_sql('dim_product', engine, if_exists='append', index=False)

dim_customer_type = df_silver[['customer_type']].drop_duplicates()
dim_customer_type.to_sql('dim_customer_type', engine, if_exists='append', index=False)

dim_payment = df_silver[['payment']].drop_duplicates().rename(columns={'payment': 'payment_method'})
dim_payment.to_sql('dim_payment', engine, if_exists='append', index=False)

print("\n Dimension Tables successfully populated!")

# Final check
with engine.connect() as conn:
    print(f"dim_branch rows: {conn.execute(text('SELECT COUNT(*) FROM dim_branch')).scalar()}")
    print(f"dim_product rows: {conn.execute(text('SELECT COUNT(*) FROM dim_product')).scalar()}")

 All old dimension tables dropped.
 Dimension tables recreated with surrogate keys (AUTO_INCREMENT).

 Dimension Tables successfully populated!
dim_branch rows: 5
dim_product rows: 8


#### PREPARE AND LOAD FACT TABLE

In [28]:
# Reload silver data to be safe
SILVER_PATH = '../data/silver/queens_supermarket_sales_silver.parquet'
df_silver = pd.read_parquet(SILVER_PATH)
print(f"Silver data loaded: {df_silver.shape}")

# ====================== FETCH SURROGATE KEYS ======================
print("Fetching surrogate keys from dimension tables...")

dim_branch = pd.read_sql("SELECT branch_key, branch_id FROM dim_branch", engine)
dim_product = pd.read_sql("SELECT product_key, product_line FROM dim_product", engine)
dim_customer = pd.read_sql("SELECT customer_type_key, customer_type FROM dim_customer_type", engine)
dim_payment = pd.read_sql("SELECT payment_key, payment_method FROM dim_payment", engine)

print(f"Mapping tables loaded → Branch: {len(dim_branch)}, Product: {len(dim_product)}")

# ====================== MERGE KEYS INTO FACT DATA ======================
df_fact = df_silver.copy()

df_fact = df_fact.merge(dim_branch[['branch_key', 'branch_id']], on='branch_id', how='left') \
                 .merge(dim_product[['product_key', 'product_line']], on='product_line', how='left') \
                 .merge(dim_customer[['customer_type_key', 'customer_type']], on='customer_type', how='left') \
                 .merge(dim_payment[['payment_key', 'payment_method']], 
                        left_on='payment', right_on='payment_method', how='left')

# Create date_key for joining with dim_date
df_fact['date_key'] = df_fact['date'].dt.strftime('%Y%m%d').astype(int)

print(f"Fact table prepared with {len(df_fact):,} rows")

# ====================== SELECT FINAL COLUMNS ======================
fact_columns = [
    'invoice_id', 'date_key', 'branch_key', 'product_key', 
    'customer_type_key', 'payment_key',
    'quantity', 'unit_price', 'subtotal', 'tax_15', 'total',
    'cogs', 'gross_income', 'gross_margin_percentage',
    'time_of_day', 'customer_segment', 'high_value_transaction', 'rating'
]

df_fact_final = df_fact[fact_columns].copy()

# ====================== LOAD INTO MySQL ======================
print("Loading data into fact_sales table... (this may take some time)")

df_fact_final.to_sql(
    name='fact_sales',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=5000,
    method='multi'
)

print(f" Fact table 'fact_sales' loaded successfully with {len(df_fact_final):,} records!")

# ====================== VERIFICATION ======================
print("\n Verification:")
with engine.connect() as conn:
    total_rows = conn.execute(text("SELECT COUNT(*) FROM fact_sales")).scalar()
    print(f"Total rows in fact_sales: {total_rows:,}")
    
    # Show sample
    sample = pd.read_sql("SELECT * FROM fact_sales LIMIT 5", engine)
    display(sample)

Silver data loaded: (50000, 30)
Fetching surrogate keys from dimension tables...
Mapping tables loaded → Branch: 5, Product: 8
Fact table prepared with 50,000 rows
Loading data into fact_sales table... (this may take some time)
 Fact table 'fact_sales' loaded successfully with 50,000 records!

 Verification:
Total rows in fact_sales: 50,000


,invoice_id,date_key,branch_key,product_key,customer_type_key,payment_key,quantity,unit_price,subtotal,tax_15,total,cogs,gross_income,gross_margin_percentage,time_of_day,customer_segment,high_value_transaction,rating
0,INV100000,20250918,1,1,1,1,5,265.40,1327.00,199.05,1526.05,887.41,638.64,41.85,Afternoon,Medium,0,7.4
1,INV100001,20250421,1,2,1,2,10,221.93,2219.30,332.90,2552.20,1482.98,1069.22,41.89,Morning,Medium,0,5.8
2,INV100002,20251214,2,1,1,3,2,383.42,766.84,115.03,881.87,545.90,335.97,38.10,Afternoon,Small,0,7.6
3,INV100003,20250706,2,3,2,3,2,1170.10,2340.20,351.03,2691.23,1778.77,912.46,33.90,Morning,Medium,0,4.6
4,INV100004,20250531,1,4,1,3,6,63.50,381.00,57.15,438.15,279.91,158.24,36.12,Evening,Small,0,4.4


#### Final Verification

In [30]:
with engine.connect() as conn:
    
    # 1. List all tables and row counts
    print(" TABLE ROW COUNTS")
    print("-" * 50)
    tables = conn.execute(text("SHOW TABLES")).fetchall()
    
    summary_data = []
    for table in tables:
        table_name = table[0]
        try:
            count = conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}`")).scalar()
            summary_data.append({"Table": table_name, "Rows": f"{count:,}"})
        except:
            summary_data.append({"Table": table_name, "Rows": "Error"})
    
    summary_df = pd.DataFrame(summary_data)
    display(summary_df)

    # 2. Sample Records from Fact Table (Safe version - without sales_key)
    print("\n SAMPLE RECORDS FROM fact_sales (First 5 rows)")
    print("-" * 60)
    
    sample_fact = pd.read_sql("""
        SELECT 
            invoice_id, 
            date_key, 
            branch_key, 
            product_key,
            customer_type_key,
            payment_key,
            quantity, 
            unit_price,
            total, 
            gross_income, 
            gross_margin_percentage,
            rating
        FROM fact_sales 
        LIMIT 5
    """, engine)
    display(sample_fact)

    # 3. Key Business Metrics
    print("\n KEY BUSINESS METRICS")
    print("-" * 50)
    
    metrics = {
        "Total Sales Records": conn.execute(text("SELECT COUNT(*) FROM fact_sales")).scalar(),
        "Unique Invoices": conn.execute(text("SELECT COUNT(DISTINCT invoice_id) FROM fact_sales")).scalar(),
        "Total Revenue (ETB)": round(conn.execute(text("SELECT SUM(total) FROM fact_sales")).scalar() or 0, 2),
        "Average Transaction Value": round(conn.execute(text("SELECT AVG(total) FROM fact_sales")).scalar() or 0, 2),
        "Total Gross Income": round(conn.execute(text("SELECT SUM(gross_income) FROM fact_sales")).scalar() or 0, 2),
        "Average Gross Margin (%)": round(conn.execute(text("SELECT AVG(gross_margin_percentage) FROM fact_sales")).scalar() or 0, 2)
    }
    
    for metric, value in metrics.items():
        print(f"{metric:30}: {value}")

    # 4. Date Range Check
    print("\n DATE RANGE")
    date_range = conn.execute(text("""
        SELECT 
            MIN(d.full_date) as start_date,
            MAX(d.full_date) as end_date
        FROM fact_sales f
        JOIN dim_date d ON f.date_key = d.date_key
    """)).fetchone()
    
    print(f"Data covers from {date_range[0]} to {date_range[1]}")

print("\n" + "="*75)
print("Star Schema Implementation Completed!")
print("Note: 'sales_key' column was not created. This can be fixed later if needed.")

 TABLE ROW COUNTS
--------------------------------------------------


,Table,Rows
0,dim_branch,5
1,dim_customer_type,2
2,dim_date,365
3,dim_payment,4
4,dim_product,8
5,fact_sales,"50,000"



 SAMPLE RECORDS FROM fact_sales (First 5 rows)
------------------------------------------------------------


,invoice_id,date_key,branch_key,product_key,customer_type_key,payment_key,quantity,unit_price,total,gross_income,gross_margin_percentage,rating
0,INV100000,20250918,1,1,1,1,5,265.40,1526.05,638.64,41.85,7.4
1,INV100001,20250421,1,2,1,2,10,221.93,2552.20,1069.22,41.89,5.8
2,INV100002,20251214,2,1,1,3,2,383.42,881.87,335.97,38.10,7.6
3,INV100003,20250706,2,3,2,3,2,1170.10,2691.23,912.46,33.90,4.6
4,INV100004,20250531,1,4,1,3,6,63.50,438.15,158.24,36.12,4.4



 KEY BUSINESS METRICS
--------------------------------------------------
Total Sales Records           : 50000
Unique Invoices               : 50000
Total Revenue (ETB)           : 145976526.58
Average Transaction Value     : 2919.53
Total Gross Income            : 57169433.56
Average Gross Margin (%)      : 39.15

 DATE RANGE
Data covers from 2025-01-01 to 2025-12-31

Star Schema Implementation Completed!
Note: 'sales_key' column was not created. This can be fixed later if needed.
